In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import string

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Define parameters
n_applications = 5000
departments = ['Engineering', 'Data Science', 'Product', 'Marketing', 'Sales', 'Operations', 'Finance', 'HR']
regions = ['North America', 'Europe', 'Asia Pacific', 'Latin America']
genders = ['Male', 'Female', 'Non-binary', 'Prefer not to say']
experience_levels = ['Entry', 'Mid', 'Senior', 'Staff/Principal']
recruiters = [f'Recruiter_{i:02d}' for i in range(1, 21)]
stages = ['Application', 'Phone Screen', 'Technical Interview', 'Panel Interview', 'Final Interview', 'Offer', 'Hire']

# Additional data for feature engineering
universities = ['MIT', 'Stanford', 'UC Berkeley', 'CMU', 'Georgia Tech', 'University of Washington', 
               'Harvard', 'Yale', 'Princeton', 'Columbia', 'NYU', 'UCLA', 'USC', 'UT Austin',
               'State University', 'Community College', 'International University', 'Online University']
job_sources = ['LinkedIn', 'Indeed', 'Company Website', 'Referral', 'Recruiter Outreach', 'Job Fair', 'University Career Fair']
skills = ['Python', 'SQL', 'Machine Learning', 'JavaScript', 'React', 'AWS', 'Kubernetes', 'Tableau', 
          'Excel', 'Project Management', 'Agile', 'Communication', 'Leadership', 'Analytics']

# Gender distribution (realistic tech industry skew)
gender_weights = [0.65, 0.32, 0.02, 0.01]
experience_weights = [0.3, 0.4, 0.25, 0.05]

def introduce_data_quality_issues(value, column_type, issue_rate=0.05):
    """Introduce realistic data quality issues"""
    
    if np.random.random() > issue_rate:
        return value
    
    if column_type == 'text':
        # Common text issues
        issues = [
            lambda x: x.lower(),  # Inconsistent case
            lambda x: f" {x} ",   # Extra whitespace
            lambda x: x.replace(' ', '_'),  # Underscore instead of space
            lambda x: x + '.',    # Extra punctuation
            lambda x: x.replace('e', '3') if len(x) > 3 else x,  # Character substitution
        ]
        return np.random.choice(issues)(value)
    
    elif column_type == 'categorical':
        # Missing or inconsistent categories
        if np.random.random() < 0.3:
            return np.nan
        elif np.random.random() < 0.2:
            return value.upper()  # Case inconsistency
        return value
    
    elif column_type == 'numeric':
        # Outliers or missing values
        if np.random.random() < 0.4:
            return np.nan
        elif np.random.random() < 0.2:
            return value * np.random.uniform(10, 100)  # Unrealistic outlier
        return value
    
    return value

# Generate base applications with data quality issues
applications = []
for i in range(n_applications):
    app_id = f"APP_{i+1:05d}"
    
    # Basic demographics with potential issues
    gender = introduce_data_quality_issues(
        np.random.choice(genders, p=gender_weights), 'categorical', 0.08
    )
    
    department = introduce_data_quality_issues(
        np.random.choice(departments), 'text', 0.06
    )
    
    region = introduce_data_quality_issues(
        np.random.choice(regions), 'text', 0.04
    )
    
    experience = introduce_data_quality_issues(
        np.random.choice(experience_levels, p=experience_weights), 'categorical', 0.07
    )
    
    recruiter = np.random.choice(recruiters)
    
    # Application date with some missing values
    app_date = datetime.now() - timedelta(days=np.random.randint(1, 180))
    if np.random.random() < 0.02:  # 2% missing dates
        app_date = None
    
    # Additional fields for feature engineering
    university = introduce_data_quality_issues(
        np.random.choice(universities), 'text', 0.10
    )
    
    # GPA with realistic distribution and missing values
    gpa = np.random.normal(3.4, 0.4)
    gpa = max(2.0, min(4.0, gpa))  # Constrain to realistic range
    gpa = introduce_data_quality_issues(gpa, 'numeric', 0.15)
    
    # Years of experience (can be inconsistent with experience_level)
    if experience == 'Entry':
        years_exp = np.random.uniform(0, 2)
    elif experience == 'Mid':
        years_exp = np.random.uniform(2, 6)
    elif experience == 'Senior':
        years_exp = np.random.uniform(5, 12)
    else:  # Staff/Principal
        years_exp = np.random.uniform(8, 20)
    
    # Sometimes introduce inconsistencies
    if np.random.random() < 0.08:
        years_exp = np.random.uniform(0, 20)  # Inconsistent with level
    
    years_exp = introduce_data_quality_issues(years_exp, 'numeric', 0.12)
    
    # Job source
    source = introduce_data_quality_issues(
        np.random.choice(job_sources), 'text', 0.05
    )
    
    # Number of skills (for feature engineering)
    num_skills = np.random.randint(2, 8)
    candidate_skills = np.random.choice(skills, size=num_skills, replace=False)
    skills_text = ', '.join(candidate_skills)
    
    # Resume score (simulated ATS score with missing values)
    resume_score = np.random.normal(75, 15)
    resume_score = max(0, min(100, resume_score))
    if np.random.random() < 0.20:  # 20% missing resume scores
        resume_score = np.nan
    
    # Previous company type (for feature engineering)
    company_types = ['Startup', 'Big Tech', 'Consulting', 'Finance', 'Government', 'Non-profit', 'Other']
    prev_company_type = introduce_data_quality_issues(
        np.random.choice(company_types), 'categorical', 0.10
    )
    
    # Phone/Email (for duplicate detection practice)
    phone = f"+1-{np.random.randint(200,999)}-{np.random.randint(200,999)}-{np.random.randint(1000,9999)}"
    email = f"candidate{i+1}@{''.join(np.random.choice(list(string.ascii_lowercase), 5))}mail.com"
    
    # Introduce some duplicates (same person, multiple applications)
    if np.random.random() < 0.03:  # 3% duplicates
        duplicate_source = np.random.randint(0, max(1, i))
        phone = f"+1-{np.random.randint(200,999)}-{np.random.randint(200,999)}-{np.random.randint(1000,9999)}"
        email = f"candidate{duplicate_source+1}@{''.join(np.random.choice(list(string.ascii_lowercase), 5))}mail.com"
    
    # Sometimes phone/email are missing or malformed
    if np.random.random() < 0.05:
        phone = None
    elif np.random.random() < 0.03:
        phone = phone.replace('+1-', '').replace('-', '')  # Inconsistent format
    
    if np.random.random() < 0.02:
        email = None
    elif np.random.random() < 0.04:
        email = email.replace('@', '_at_')  # Malformed email
    
    applications.append({
        'application_id': app_id,
        'gender': gender,
        'department': department,
        'region': region,
        'experience_level': experience,
        'years_experience': years_exp,
        'recruiter': recruiter,
        'application_date': app_date,
        'university': university,
        'gpa': gpa,
        'source': source,
        'skills': skills_text,
        'num_skills': num_skills,
        'resume_score': resume_score,
        'previous_company_type': prev_company_type,
        'phone': phone,
        'email': email
    })

df_apps = pd.DataFrame(applications)

# Define stage-specific conversion rates with bias patterns
def get_conversion_rate(stage, gender, department, experience, region, gpa=None, resume_score=None):
    """Calculate conversion rates with realistic bias patterns"""
    
    base_rates = {
        'Phone Screen': 0.35,
        'Technical Interview': 0.70,
        'Panel Interview': 0.65,
        'Final Interview': 0.80,
        'Offer': 0.75,
        'Hire': 0.85
    }
    
    rate = base_rates.get(stage, 1.0)
    
    # GPA influence (for feature engineering insights)
    if gpa and not pd.isna(gpa):
        if gpa >= 3.7:
            rate *= 1.15
        elif gpa <= 2.8:
            rate *= 0.85
    
    # Resume score influence
    if resume_score and not pd.isna(resume_score):
        if resume_score >= 85:
            rate *= 1.20
        elif resume_score <= 60:
            rate *= 0.80
    
    # Gender bias patterns
    if gender == 'Female':
        if department in ['Engineering', 'Data Science']:
            if stage in ['Technical Interview', 'Panel Interview']:
                rate *= 0.85
            elif stage == 'Offer':
                rate *= 0.90
        elif department in ['Marketing', 'HR']:
            rate *= 1.10
    
    elif gender == 'Non-binary' or gender == 'Prefer not to say':
        if stage in ['Panel Interview', 'Final Interview']:
            rate *= 0.88
    
    # Experience bias
    if experience == 'Entry':
        if stage in ['Technical Interview', 'Panel Interview']:
            rate *= 0.80
    elif experience in ['Staff/Principal']:
        rate *= 1.15
    
    # Regional differences
    if region == 'Asia Pacific':
        if stage == 'Phone Screen':
            rate *= 0.90
    elif region == 'Latin America':
        if stage in ['Panel Interview', 'Final Interview']:
            rate *= 0.95
    
    # Department-specific patterns
    if department == 'Engineering':
        if stage == 'Technical Interview':
            rate *= 0.80
    elif department == 'Sales':
        if stage == 'Final Interview':
            rate *= 1.20
    
    # Add randomness
    rate *= np.random.uniform(0.85, 1.15)
    
    return min(1.0, max(0.05, rate))

# Generate funnel progression with additional data quality issues
funnel_data = []
current_candidates = df_apps.to_dict('records')

for stage_idx, stage in enumerate(stages):
    if stage == 'Application':
        for candidate in current_candidates:
            funnel_data.append({
                **candidate,
                'stage': stage,
                'stage_date': candidate['application_date'],
                'days_in_stage': 0,
                'outcome': 'Advanced',
                'interviewer_rating': np.nan,
                'interview_notes': np.nan
            })
    else:
        next_candidates = []
        
        for candidate in current_candidates:
            conversion_rate = get_conversion_rate(
                stage, 
                candidate['gender'], 
                candidate['department'],
                candidate['experience_level'],
                candidate['region'],
                candidate.get('gpa'),
                candidate.get('resume_score')
            )
            
            advanced = np.random.random() < conversion_rate
            
            # Calculate stage timing with some outliers
            if stage == 'Phone Screen':
                days_in_stage = max(1, int(np.random.normal(6, 2)))
            elif stage == 'Technical Interview':
                days_in_stage = max(1, int(np.random.normal(10, 4)))
            elif stage in ['Panel Interview', 'Final Interview']:
                days_in_stage = max(1, int(np.random.normal(14, 6)))
            elif stage == 'Offer':
                days_in_stage = max(1, int(np.random.normal(5, 2)))
            else:  # Hire
                days_in_stage = max(1, int(np.random.normal(15, 8)))
            
            # Some extreme outliers for data quality issues
            if np.random.random() < 0.02:
                days_in_stage = np.random.randint(60, 200)
            
            stage_date = candidate.get('stage_date', candidate['application_date'])
            if stage_date:
                stage_date = stage_date + timedelta(days=days_in_stage)
            
            # Interviewer rating (1-5 scale) with missing values
            interviewer_rating = np.nan
            interview_notes = np.nan
            
            if stage in ['Phone Screen', 'Technical Interview', 'Panel Interview', 'Final Interview']:
                if np.random.random() < 0.70:  # 70% have ratings
                    if advanced:
                        interviewer_rating = np.random.choice([3, 4, 5], p=[0.2, 0.5, 0.3])
                    else:
                        interviewer_rating = np.random.choice([1, 2, 3], p=[0.4, 0.4, 0.2])
                
                # Some interview notes (text data for NLP feature engineering)
                if np.random.random() < 0.40:  # 40% have notes
                    notes_templates = [
                        "Strong technical skills, good communication",
                        "Needs improvement in problem solving",
                        "Excellent cultural fit, very enthusiastic",
                        "Solid experience but lacks leadership",
                        "Outstanding candidate, highly recommend",
                        "Average performance, meets requirements",
                        "Concerns about technical depth"
                    ]
                    interview_notes = np.random.choice(notes_templates)
                    # Add some text quality issues
                    if np.random.random() < 0.1:
                        interview_notes = interview_notes.lower()
                    if np.random.random() < 0.05:
                        interview_notes = interview_notes + " !!!"
            
            funnel_data.append({
                **candidate,
                'stage': stage,
                'stage_date': stage_date,
                'days_in_stage': days_in_stage,
                'outcome': 'Advanced' if advanced else 'Rejected',
                'interviewer_rating': interviewer_rating,
                'interview_notes': interview_notes
            })
            
            if advanced:
                candidate['stage_date'] = stage_date
                next_candidates.append(candidate)
        
        current_candidates = next_candidates

# Create final dataset
df_funnel = pd.DataFrame(funnel_data)

# Add some additional columns with data quality issues
df_funnel['days_since_application'] = np.nan
mask = df_funnel['stage_date'].notna() & df_funnel['application_date'].notna()
df_funnel.loc[mask, 'days_since_application'] = (
    df_funnel.loc[mask, 'stage_date'] - df_funnel.loc[mask, 'application_date']
).dt.days

# Add quarters and months (with some missing due to missing dates)
df_funnel['quarter'] = df_funnel['application_date'].dt.to_period('Q')
df_funnel['month'] = df_funnel['application_date'].dt.to_period('M')

# Add salary data with more realistic issues
salary_ranges = {
    'Engineering': {'Entry': (95000, 120000), 'Mid': (120000, 160000), 'Senior': (160000, 220000), 'Staff/Principal': (220000, 350000)},
    'Data Science': {'Entry': (90000, 115000), 'Mid': (115000, 155000), 'Senior': (155000, 210000), 'Staff/Principal': (210000, 320000)},
    'Product': {'Entry': (85000, 110000), 'Mid': (110000, 145000), 'Senior': (145000, 190000), 'Staff/Principal': (190000, 280000)},
    'Marketing': {'Entry': (60000, 80000), 'Mid': (80000, 110000), 'Senior': (110000, 150000), 'Staff/Principal': (150000, 220000)},
    'Sales': {'Entry': (55000, 75000), 'Mid': (75000, 105000), 'Senior': (105000, 140000), 'Staff/Principal': (140000, 200000)},
    'Operations': {'Entry': (65000, 85000), 'Mid': (85000, 115000), 'Senior': (115000, 155000), 'Staff/Principal': (155000, 230000)},
    'Finance': {'Entry': (70000, 90000), 'Mid': (90000, 125000), 'Senior': (125000, 170000), 'Staff/Principal': (170000, 260000)},
    'HR': {'Entry': (60000, 80000), 'Mid': (80000, 110000), 'Senior': (110000, 150000), 'Staff/Principal': (150000, 220000)}
}

def generate_salary(department, experience, gender, region):
    # Handle missing/malformed department or experience
    if pd.isna(department) or department not in salary_ranges:
        department = 'Operations'  # Default
    if pd.isna(experience) or experience not in salary_ranges[department]:
        experience = 'Mid'  # Default
    
    base_min, base_max = salary_ranges[department][experience]
    
    # Gender pay gap
    if gender == 'Female':
        multiplier = np.random.uniform(0.92, 0.98)
    else:
        multiplier = np.random.uniform(0.98, 1.02)
    
    # Regional adjustments
    regional_multipliers = {
        'North America': 1.0,
        'Europe': 0.85,
        'Asia Pacific': 0.70,
        'Latin America': 0.60
    }
    
    # Handle missing/malformed region
    if pd.isna(region) or region not in regional_multipliers:
        regional_mult = 1.0
    else:
        regional_mult = regional_multipliers[region]
    
    adjusted_min = base_min * multiplier * regional_mult
    adjusted_max = base_max * multiplier * regional_mult
    
    return int(np.random.uniform(adjusted_min, adjusted_max))

# Add salary with missing values and some data entry errors
df_funnel['salary_offered'] = np.nan
df_funnel['salary_accepted'] = np.nan

offer_mask = df_funnel['stage'].isin(['Offer', 'Hire'])
for idx in df_funnel[offer_mask].index:
    row = df_funnel.loc[idx]
    
    # 10% missing salary data
    if np.random.random() < 0.10:
        continue
        
    salary = generate_salary(row['department'], row['experience_level'], row['gender'], row['region'])
    
    # Some data entry errors (extra zeros, missing zeros)
    if np.random.random() < 0.02:
        salary = salary * 10  # Extra zero
    elif np.random.random() < 0.01:
        salary = salary // 10  # Missing zero
    
    df_funnel.loc[idx, 'salary_offered'] = salary
    
    if row['stage'] == 'Hire':
        negotiation_factor = np.random.uniform(0.98, 1.05)
        df_funnel.loc[idx, 'salary_accepted'] = int(salary * negotiation_factor)

# Add some completely duplicate rows (data quality issue)
if len(df_funnel) > 100:
    duplicate_indices = np.random.choice(df_funnel.index, size=int(len(df_funnel) * 0.01), replace=False)
    duplicate_rows = df_funnel.loc[duplicate_indices].copy()
    df_funnel = pd.concat([df_funnel, duplicate_rows], ignore_index=True)

# Introduce some inconsistent data types
df_funnel['gpa'] = df_funnel['gpa'].astype('object')  # Mixed types for cleaning practice

# Reorder columns
column_order = [
    'application_id', 'stage', 'outcome', 'gender', 'department', 'region', 
    'experience_level', 'years_experience', 'university', 'gpa', 'recruiter', 
    'source', 'skills', 'num_skills', 'resume_score', 'previous_company_type',
    'phone', 'email', 'application_date', 'stage_date', 'days_in_stage', 
    'days_since_application', 'interviewer_rating', 'interview_notes',
    'salary_offered', 'salary_accepted', 'quarter', 'month'
]

df_funnel = df_funnel[column_order]

# Save to CSV
df_funnel.to_csv('recruiting_funnel_data_messy.csv', index=False)

print("Dataset created successfully with data quality issues!")
print(f"Total records: {len(df_funnel)}")
print(f"Unique applications: {df_funnel['application_id'].nunique()}")

# Data Quality Report
print("\n=== DATA QUALITY ISSUES TO IDENTIFY ===")
print(f"1. Missing values in critical fields:")
for col in ['gender', 'department', 'application_date', 'gpa', 'resume_score']:
    missing_pct = (df_funnel[col].isna().sum() / len(df_funnel)) * 100
    print(f"   - {col}: {missing_pct:.1f}% missing")

print(f"\n2. Potential duplicates: {len(df_funnel) - df_funnel['application_id'].nunique()} duplicate application IDs")

print(f"\n3. Text inconsistencies in department:")
print(df_funnel['department'].value_counts().head(10))

print(f"\n4. Outliers in salary_offered:")
valid_salaries = df_funnel['salary_offered'].dropna()
if len(valid_salaries) > 0:
    print(f"   - Min: ${valid_salaries.min():,.0f}")
    print(f"   - Max: ${valid_salaries.max():,.0f}")
    print(f"   - Median: ${valid_salaries.median():,.0f}")

print(f"\n5. Inconsistent phone/email formats:")
invalid_phones = df_funnel['phone'].dropna().str.contains(r'^(?!\+1-\d{3}-\d{3}-\d{4}$)', na=False).sum()
invalid_emails = df_funnel['email'].dropna().str.contains(r'[^@]+@[^@]+\.[^@]+', na=False) == False
print(f"   - Invalid phone formats: {invalid_phones}")
print(f"   - Invalid email formats: {invalid_emails.sum()}")

print("\n=== FEATURE ENGINEERING OPPORTUNITIES ===")
print("1. Create 'is_top_university' flag from university names")
print("2. Extract skill categories from skills text")
print("3. Calculate 'time_to_offer' and 'time_to_hire' metrics")
print("4. Create experience_consistency flag (years_experience vs experience_level)")
print("5. Engineer 'high_performer' flag from GPA + resume_score")
print("6. Extract domain from email for company type analysis")
print("7. Create funnel stage conversion rates by recruiter")
print("8. Text mining on interview_notes for sentiment/themes")
print("9. Create 'weekend_applicant' flag from application_date")
print("10. Calculate recruiter workload metrics")

print("\n=== SAMPLE ANALYSIS QUESTIONS ===")
print("- How do data quality issues correlate with conversion rates?")
print("- Does missing GPA data indicate bias in our screening?") 
print("- Which recruiters have the most incomplete data?")
print("- How does resume_score predict actual performance?")
print("- Are there patterns in missing salary data by gender/department?")

Dataset created successfully with data quality issues!
Total records: 15011
Unique applications: 5000

=== DATA QUALITY ISSUES TO IDENTIFY ===
1. Missing values in critical fields:
   - gender: 2.3% missing
   - department: 0.0% missing
   - application_date: 2.3% missing
   - gpa: 6.3% missing
   - resume_score: 19.6% missing

2. Potential duplicates: 10011 duplicate application IDs

3. Text inconsistencies in department:
department
HR               1901
Marketing        1832
Sales            1828
Finance          1774
Product          1771
Data Science     1767
Engineering      1749
Operations       1747
finance            43
Data Science.      39
Name: count, dtype: int64

4. Outliers in salary_offered:
   - Min: $5,570
   - Max: $2,245,850
   - Median: $91,089

5. Inconsistent phone/email formats:
   - Invalid phone formats: 360
   - Invalid email formats: 495

=== FEATURE ENGINEERING OPPORTUNITIES ===
1. Create 'is_top_university' flag from university names
2. Extract skill catego